# Level 2 — Vision Transformers (ViT-S/16, 선택적으로 Swin-Tiny)

**목표**: ViT (와 선택적으로 Swin-T) 를 직접 구현하고 Multi-task 로 연결하여 Level 1 의 CNN 들과 비교합니다.

**Pretrained 가중치**: ImageNet `.pth` 파일을 본인이 구현한 모델의 `state_dict` 에 로드하는 것은 허용됩니다. 출처를 명시하세요. **`timm` / `torchvision.models` import 는 금지** 입니다.

In [1]:
import os
import sys

repo_url  = "https://github.com/min0712-cdl/HYU-AUE8088-PA2.git"
repo_name = "HYU-AUE8088-PA2"

if not os.path.exists(f"/content/{repo_name}"):
    !git clone {repo_url}
else:
    # 런타임이 살아있고 이미 클론된 경우 최신 코드로 업데이트
    !git -C /content/{repo_name} pull

%cd /content/{repo_name}

from google.colab import drive
drive.mount("/content/drive")

CHECKPOINT_DIR = "/content/drive/MyDrive/AUE8088_PA2/checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

%load_ext autoreload
%autoreload 2

# 의존성 설치 (이미 설치된 패키지는 빠르게 skip)
!pip install -q -r requirements.txt

Cloning into 'HYU-AUE8088-PA2'...
remote: Enumerating objects: 57, done.
remote: Counting objects: 100% (57/57), done.
remote: Compressing objects: 100% (45/45), done.
remote: Total 57 (delta 19), reused 49 (delta 11), pack-reused 0 (from 0)
Receiving objects: 100% (57/57), 59.33 KiB | 6.59 MiB/s, done.
Resolving deltas: 100% (19/19), done.
/content/HYU-AUE8088-PA2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 274.9/274.9 kB 31.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.2/455.2 kB 59.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.4/26.4 MB 86.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 102.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 75.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 41.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
import torch
from torch import nn
from torch.utils.data import DataLoader

from src.utils.seed import set_seed, seed_worker
from src.utils.transforms import train_transform, eval_transform
from src.utils.trainer import MultiTaskTrainer, TrainConfig
from src.utils.wandb_logger import WandbLogger
from src.utils.metrics import collect_predictions, confusion_matrices, CLASS_NAMES
from src.datasets.bdd_attr import BDDAttrDataset, ATTRIBUTES
from src.models.vit import vit_small_patch16_224
# from src.models.swin import SwinTiny  # 선택 사항

SEED = 42
set_seed(SEED, deterministic=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
import wandb; wandb.login()   # API key 입력

# wandb 설정 — 비활성화하려면 None
WANDB_PROJECT = "aue8088-pa2"
WANDB_TAGS    = ["level2"]

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: min0712-cdl (min0712-cdl-hanyang-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [4]:
DATA_ROOT = "../data/set_a"
BATCH = 64

# --- 데이터셋 자동 다운로드 (Google Drive) ---------------------------------
# ../data/set_a 가 없으면 zip 을 받아 상위 폴더에 압축 해제 → ../data/set_a, ../data/set_b 생성.
import os, sys, zipfile, subprocess

GDRIVE_FILE_ID = "1L7YC70QlO87aIbE5lbtQ94HUINJijBKK"
ZIP_PATH   = "../aue8088_pa2_data.zip"
EXTRACT_TO = ".."   # zip 내부 최상위가 data/ 이므로 상위 폴더에 풀면 ../data/... 가 됨

if not os.path.isdir(DATA_ROOT):
    try:
        import gdown
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
        import gdown

    if not os.path.exists(ZIP_PATH):
        print("데이터셋 zip 다운로드 중...")
        gdown.download(id=GDRIVE_FILE_ID, output=ZIP_PATH, quiet=False)

    print("압축 해제 중...")
    with zipfile.ZipFile(ZIP_PATH) as zf:
        zf.extractall(EXTRACT_TO)
    print(f"완료 → {DATA_ROOT}")
else:
    print(f"데이터셋이 이미 존재합니다 → {DATA_ROOT}")
# --------------------------------------------------------------------------

train_ds = BDDAttrDataset(DATA_ROOT, "train", transform=train_transform())
val_ds   = BDDAttrDataset(DATA_ROOT, "val",   transform=eval_transform())

g = torch.Generator(); g.manual_seed(SEED)
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  num_workers=2, worker_init_fn=seed_worker, generator=g, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)

데이터셋 zip 다운로드 중...


Downloading...
From (original): https://drive.google.com/uc?id=1L7YC70QlO87aIbE5lbtQ94HUINJijBKK
From (redirected): https://drive.google.com/uc?id=1L7YC70QlO87aIbE5lbtQ94HUINJijBKK&confirm=t&uuid=b9a32dd9-f862-4f7d-be04-47e658dfc7b8
To: /content/aue8088_pa2_data.zip
100%|██████████| 243M/243M [00:02<00:00, 98.1MB/s] 


압축 해제 중...
완료 → ../data/set_a


In [5]:
# 선택 사항: 본인 ViT 구현체에 ImageNet pretrained 가중치를 로드하는 절차
#
# 진행 방식:
#   1) 공개된 ViT-S/16 체크포인트 (.pth) 를 다운로드.
#   2) 모델 인스턴스 생성:  model = vit_small_patch16_224()
#   3) 키 매핑 후 로드:
#        pre = torch.load('vit_s16.pth')
#        missing, unexpected = model.load_state_dict(remap(pre), strict=False)
#        # Multi-task head 는 task 종속이므로 random init 유지.
#
# 사용한 체크포인트 출처와 매칭된 키 개수를 리포트에 기재하세요.
DEIT_SMALL_URL = (
    "https://dl.fbaipublicfiles.com/deit/"
    "deit_small_patch16_224-cd65a155.pth"
)

def build_vit(use_pretrained):
    model = vit_small_patch16_224()
    info = {"source": None, "matched_tensors": 0, "matched_parameters": 0}

    if not use_pretrained:
        return model.to(device), info

    checkpoint = torch.hub.load_state_dict_from_url(
        DEIT_SMALL_URL, map_location="cpu", check_hash=True
    )
    pretrained = checkpoint["model"]
    target = model.state_dict()
    remapped = {}

    for key, value in pretrained.items():
        key = key.replace(".mlp.fc1.", ".mlp.0.")
        key = key.replace(".mlp.fc2.", ".mlp.3.")
        if key.startswith("head."):
            continue
        if key in target and target[key].shape == value.shape:
            remapped[key] = value

    missing, unexpected = model.load_state_dict(remapped, strict=False)
    non_head_missing = [key for key in missing if not key.startswith("head.")]
    if non_head_missing or unexpected:
        raise RuntimeError(
            f"Pretrained backbone mismatch: missing={non_head_missing}, "
            f"unexpected={unexpected}"
        )

    info = {
        "source": DEIT_SMALL_URL,
        "matched_tensors": len(remapped),
        "matched_parameters": sum(value.numel() for value in remapped.values()),
    }
    print(f"Matched pretrained tensors: {info['matched_tensors']}")
    print(f"Matched pretrained parameters: {info['matched_parameters']:,}")
    print(f"Random-init task head keys: {missing}")
    return model.to(device), info

In [6]:
epochs = 30

def train_vit(use_pretrained):
    set_seed(SEED, deterministic=True)
    g.manual_seed(SEED)
    model, pretrain_info = build_vit(use_pretrained)
    optim = torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=5e-2)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(optim, T_max=epochs)
    losses = {a: nn.CrossEntropyLoss() for a in ATTRIBUTES}
    variant = "pretrained" if use_pretrained else "scratch"

    logger = WandbLogger(
        project=WANDB_PROJECT,
        run_name=f"level2-vit_s16-{variant}",
        config={
            "backbone": "vit_s16", "pretrained": use_pretrained,
            "pretrained_source": pretrain_info["source"],
            "matched_tensors": pretrain_info["matched_tensors"],
            "matched_parameters": pretrain_info["matched_parameters"],
            "epochs": epochs, "batch": BATCH, "lr": 5e-4, "weight_decay": 5e-2, "seed": SEED,
        },
        tags=WANDB_TAGS + ["vit_s16", variant],
    )
    trainer = MultiTaskTrainer(
        model, optim, sched, losses, device,
        TrainConfig(epochs=epochs), logger=logger
    )

    # TODO: trainer.fit(train_loader, val_loader)
    history = trainer.fit(train_loader, val_loader)

    # 학습 종료 후:
    val_pred, _, val_tgt, _ = collect_predictions(model, val_loader, device)
    cms = confusion_matrices(val_pred, val_tgt)
    for a in ATTRIBUTES:
        logger.log_confusion_matrix(f"final/cm_{a}", cms[a], CLASS_NAMES[a])
    logger.finish()
    torch.save(
        model.state_dict(),
        os.path.join(CHECKPOINT_DIR, f"level2_vit_s16_{variant}.pth"),
    )
    model = model.cpu()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return model, history

scratch_model, scratch_history = train_vit(use_pretrained=False)
pretrained_model, pretrained_history = train_vit(use_pretrained=True)

wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


[epoch 01/30] train_loss=2.3619  val_avg_MF1=0.3884  per={'weather': 0.20039308679954712, 'scene': 0.3264927196325613, 'timeofday': 0.6383568122698557}


[epoch 02/30] train_loss=2.0888  val_avg_MF1=0.4128  per={'weather': 0.1992101016923712, 'scene': 0.3979669261359402, 'timeofday': 0.6410988367510106}


[epoch 03/30] train_loss=2.0346  val_avg_MF1=0.4001  per={'weather': 0.22857895846510648, 'scene': 0.33702255058187264, 'timeofday': 0.6347213415063764}


[epoch 04/30] train_loss=2.0100  val_avg_MF1=0.4146  per={'weather': 0.2372021843031188, 'scene': 0.34028620774734764, 'timeofday': 0.6664137353906331}


...

[epoch 27/30] train_loss=0.0245  val_avg_MF1=0.6718  per={'weather': 0.5473717272224735, 'scene': 0.6582883076175778, 'timeofday': 0.8097193521609354}


[epoch 28/30] train_loss=0.0227  val_avg_MF1=0.6646  per={'weather': 0.5500380695173263, 'scene': 0.6454000240144198, 'timeofday': 0.7984303922188136}


[epoch 29/30] train_loss=0.0246  val_avg_MF1=0.6591  per={'weather': 0.540228248387593, 'scene': 0.6355295962166191, 'timeofday': 0.8016072547559087}


[epoch 30/30] train_loss=0.0209  val_avg_MF1=0.6589  per={'weather': 0.540228248387593, 'scene': 0.6348619873664315, 'timeofday': 0.8016072547559087}


epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
lr,█████▇▇▇▇▆▆▆▅▅▅▄▄▃▃▃▂▂▂▂▁▁▁▁▁▁
train/loss,█▇▆▆▆▅▅▅▅▄▄▄▃▃▃▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁
val/avg_macro_f1,▁▄▃▇▄▆▇▇▇▇▇█▇▇▇▇▆▇▇▆▇█▇▇████▇▇
val/mf1_scene,▄▂▁▆▂▅▇▆█▆▇▇█▇▇▇▆▆▇▆▆██▆███▇▇▇
val/mf1_timeofday,▁▇▇█▇▇▇▇▆▆▅▆▆▆▆▄▅▅▅▆▆▆▅▆▆▆▆▆▆▆
val/mf1_weather,▁▄▄▆▅▆▆▇▆▇▇█▆▆▇▆▇▇▇▆▇▇▇▇▇▇▇▇▇▇
epoch,30
lr,0
train/loss,0.02087
val/avg_macro_f1,0.6589


## 분석 (리포트 필수 포함 항목)

1. **CNN vs Transformer**: 동일 epoch 예산 하에서 ResNet-50 (Level 1) 과 ViT-S (Level 2) 의 Avg-MF1 을 비교하세요.
2. **Pretrained vs Scratch**: 약 5천 장 규모의 소규모 데이터셋에서 ImageNet 초기화가 실제로 얼마나 도움이 되는지 정량적으로 보고하세요.
3. **속성별 거동**: ViT 가 ResNet 대비 Weather 와 Time of Day 사이의 오류 분포를 다르게 가져가는지, 그 원인을 가설로 제시하세요.